# Angel Stadium Game & Weather Dataset (2018-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Angel Stadium. It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Angel Stadium'
HOME_TEAM = 'LAA'
SEASONS = range(2018, 2026)  # 2018 through 2025
TIMEZONE = 'America/Los_Angeles'

# Coordinates
STADIUM_LAT = 33.799925
STADIUM_LON = -117.883194

# Outfield directions (degrees from north)
CF_DIR = 45.0    # Center field: NE
LCF_DIR = 25.0   # Left-center field: NNE
RCF_DIR = 65.0   # Right-center field: ENE

# Output file
OUTPUT_FILE = 'angels_data_2018.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Angel Stadium
Home team: LAA
Seasons: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Timezone: America/Los_Angeles


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Angel Stadium home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Angel Stadium games: 597
Seasons: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2018    81
2019    81
2020    30
2021    81
2022    81
2023    81
2024    81
2025    81
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   529461 2018-04-02    2018       CLE                 0                 6           6              3          15      8    13            318               81.5          2     52       0.0385   
1   529475 2018-04-03    2018       CLE                13                 2          15              6          17      7    14            296               85.3          7     48       0.1458   
2   529486 2018-04-04    2018       CLE                 3                 2           5              2          25     10    13            408 

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}\u00b0C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}\u00b0")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 26.1°C
Sample wind: 13.4 km/h from 234°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 597
Missing temp data: 0
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  14.133333  84.666667  1006.300000   0.0   4.233333  187.070523   529461
1  15.066667  79.000000  1009.733333   0.0   6.566667  238.059233   529475
2  20.866667  54.666667  1010.633333   0.0  12.066667  228.333363   529486


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Angel Stadium outfield directions (degrees from north):
- Center field: ~45° (NE)
- Left-center field: ~25° (NNE)
- Right-center field: ~65° (ENE)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  597.000000  597.000000  597.000000
mean     9.333444    9.469203    8.071934
std      3.930232    3.592800    4.234528
min     -6.228932   -8.213191   -8.090875
25%      6.834033    7.165864    5.308153
50%      8.870861    9.297633    7.372121
75%     10.845843   11.423571    9.802398
max     23.460699   20.990329   29.066528


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

angels_data = games_full[final_columns].copy()
angels_data = angels_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    angels_data[col] = angels_data[col].round(decimals)

print(f"Final dataset: {angels_data.shape[0]} rows x {angels_data.shape[1]} columns")

Final dataset: 597 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = angels_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games (wider range for doubleheaders)
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(angels_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = angels_data[col].isna().sum()
    pct = 100 * n_null / len(angels_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', angels_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', angels_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', angels_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', angels_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', angels_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', angels_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', angels_data['temp_f'].mean(), '~65-80 F'),
    ('Min game temp', angels_data['temp_f'].min(), '>45 F'),
    ('Max game temp', angels_data['temp_f'].max(), '<110 F'),
    ('Avg wind speed (km/h)', angels_data['wspd'].mean(), '~5-15 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(angels_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 30 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  2025: 81 games [OK] (expected 75-100)
  TOTAL: 597 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 2 nulls (0.3%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 9.37 (expected ~8-10)
  Avg HR/game: 2.72 (expected ~2-3)
  Avg K/game: 17.66 (expected ~16-18)
  Avg BB/game: 6.78 (expected ~6-7)
  Avg exit velocity: 

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
angels_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,529461,2018-04-02,2018,CLE,2018-04-02 19:07:00,19,0,6,6,3,15,8,13,318,81.5,2,52,0.0385,0.2308,57.4,14.1,84.7,1006.3,0.0,4.2,2.6,187.1,S,3.34,4.03,2.25
1,529475,2018-04-03,2018,CLE,2018-04-03 19:07:00,19,13,2,15,6,17,7,14,296,85.3,7,48,0.1458,0.4286,59.1,15.1,79.0,1009.7,0.0,6.6,4.1,238.1,SW,6.40,5.50,6.52
2,529486,2018-04-04,2018,CLE,2018-04-04 13:07:00,13,3,2,5,2,25,10,13,408,84.3,4,61,0.0656,0.1538,69.6,20.9,54.7,1010.6,0.0,12.1,7.5,228.3,SW,12.05,11.08,11.56
3,529514,2018-04-06,2018,ATH,2018-04-06 19:07:00,19,13,9,22,7,12,6,27,329,87.7,8,64,0.1250,0.2593,59.8,15.5,87.7,1009.3,0.0,10.0,6.2,173.3,S,6.22,8.54,3.16
4,529527,2018-04-07,2018,ATH,2018-04-07 18:07:00,18,3,7,10,3,19,12,14,329,82.7,3,45,0.0667,0.2143,62.3,16.8,85.0,1008.0,0.0,13.0,8.1,187.3,S,10.29,12.39,6.95
5,529542,2018-04-08,2018,ATH,2018-04-08 13:07:00,13,6,1,7,3,20,6,10,275,84.0,3,40,0.0750,0.3000,75.9,24.4,39.0,1007.2,0.0,9.3,5.8,253.0,W,8.18,6.20,9.18
6,529655,2018-04-17,2018,BOS,2018-04-17 19:07:00,19,1,10,11,6,13,11,19,313,84.7,7,56,0.1250,0.3158,60.4,15.8,33.7,1011.8,0.0,3.9,2.4,97.0,E,-2.40,-1.21,-3.31
7,529670,2018-04-18,2018,BOS,2018-04-18 19:07:00,19,0,9,9,3,20,4,20,323,83.4,2,54,0.0370,0.1500,56.1,13.4,80.3,1010.3,0.0,10.7,6.6,183.3,S,7.99,9.94,5.08
8,529684,2018-04-19,2018,BOS,2018-04-19 19:07:00,19,2,8,10,3,14,6,18,307,84.2,7,56,0.1250,0.1667,55.0,12.8,73.0,1011.1,0.0,10.7,6.6,142.5,SE,1.40,4.93,-2.30
9,529702,2018-04-20,2018,SF,2018-04-20 19:07:00,19,1,8,9,4,16,6,14,280,82.5,5,51,0.0980,0.2857,59.3,15.2,74.3,1013.2,0.0,4.9,3.0,183.1,S,3.62,4.52,2.29


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
angels_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(angels_data)}, Columns: {len(angels_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == angels_data.shape, f"Shape mismatch: {verify.shape} vs {angels_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/angels_data_2018.csv
File size: 89.1 KB
Rows: 597, Columns: 31

Save & reload verification: PASSED
